# 05. 채점 · 분석 · 표와 그림 (분석 담당)

드라이브 `outputs/`에 모인 모든 결과를 같은 코드로 채점한다. 일부만 끝난 상태에서도 실행할 수 있고, 새 결과가 올라오면 다시 실행하면 된다 (채점 결과는 `results/cache/`에 저장돼 재사용).

In [ ]:
# @title 0. 공통 준비 (모든 노트북 동일)
import sys
if any(m.startswith("msrs_paper") for m in sys.modules):
    raise RuntimeError("이전에 불러온 코드가 메모리에 남아 있습니다. 메뉴 '런타임 → 세션 다시 시작' 후 이 셀부터 다시 실행하세요.")
from google.colab import drive
drive.mount("/content/drive")

REPO_URL = "https://github.com/yuhwani/multimodal-recsys.git"  # upstream에 머지되면 3sunghynix 주소로 변경
BRANCH = "main"
!rm -rf /content/repo && git clone -q -b {BRANCH} {REPO_URL} /content/repo
!pip -q install -r /content/repo/paper/requirements.txt
!cd /content/repo && git log -1 --format="코드 커밋: %h %s (%cd)"

import sys
sys.path.insert(0, "/content/repo/paper")
from msrs_paper.config import load_all, Paths
cfg, exps, plan = load_all()
DRIVE_ROOT = cfg["drive_root"]  # 공유 폴더 바로가기 경로가 다르면 이 줄만 바꾼다
paths = Paths(DRIVE_ROOT).make_dirs()
print("드라이브 폴더:", paths.root)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# @title 1. 공유 준비물 확인 (체크섬이 다르면 멈춤)
from msrs_paper.utils import verify_manifest
m = verify_manifest(paths)
print(f"준비물 {len(m['files'])}개 일치 — 준비물 커밋 {m['git_commit']}")

In [ ]:
# @title 2. 전체 채점과 분석
from msrs_paper import analysis
t1, contrib, overall, mt = analysis.run_all(cfg, exps, plan, paths)
print("주 검정:", mt)
t1

In [ ]:
# @title 3. 기여도 표 (주 검정 조건)
m = plan["main_test"]
contrib[(contrib["measure"] == m["measure"]) & (contrib["segment"] == m["segment"])]

In [ ]:
# @title 4. 전체 이미지 기여도 (인코더 × 구간, 부트스트랩 95% 구간)
overall

In [ ]:
# @title 5. 그림 1 · 그림 2
from IPython.display import Image, display
display(Image(str(paths.results / "fig1.png")))
display(Image(str(paths.results / "fig2.png")))

## 라벨 표본 검사 (한 번만)
외관 그룹별 이미지 40장 격자와 확인표(`results/label_check/label_check.csv`)를 만든다. 격자를 보고 csv의 `label_correct`에 1/0을 채운 뒤 일치율을 논문에 적는다.

In [ ]:
# @title 6. 라벨 표본 검사 자료 만들기
import shutil
from msrs_paper.data_prep import load_items
from msrs_paper.inspect_images import label_sample
IMG_ZIP = "/content/images_subset.zip"
shutil.copy(paths.raw / "images_subset.zip", IMG_ZIP)
out = paths.results / "label_check"; out.mkdir(parents=True, exist_ok=True)
label_sample(load_items(paths), IMG_ZIP, out)
!ls "{out}"